# TetraFT — Build FineWeb-Edu 50M sample (Kaggle)

**Goal:** Stream `HuggingFaceFW/fineweb-edu` → fixed `train.jsonl` + held-out `val.jsonl`.

| Setting | Value |
|---------|--------|
| Accelerator | **None** (CPU) |
| Internet | **ON** |
| Train budget | 50M whitespace-token estimate |
| Val budget | 0.5M |
| Seed | 42 |
| Expected wall time | ~1–3 h (up to ~6 h if HF is slow) |

**Prereq:** Attach Dataset `tetraft-code` (flat repo `.py` files) **or** upload `data.py` into this notebook's input.

**After success:** Save Version → include output → create Dataset `tetraft-fineweb-edu-50m` from `/kaggle/working/fineweb-50m/`.

In [ ]:
# Install streaming deps (internet ON)
%pip install -q datasets huggingface_hub

In [ ]:
import os
import sys
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)

# Resolve tetraft-code (Kaggle flattens or nests under /kaggle/input/<slug>/)
CANDIDATES = [
    Path("/kaggle/input/tetraft-code"),
    Path("/kaggle/working"),  # if you copied .py here
    Path("."),
]
# Also scan one level under /kaggle/input
inp = Path("/kaggle/input")
if inp.is_dir():
    for sub in sorted(inp.iterdir()):
        if sub.is_dir():
            CANDIDATES.append(sub)

code_root = None
for root in CANDIDATES:
    if (root / "data.py").is_file():
        code_root = root
        break

if code_root is None:
    raise FileNotFoundError(
        "data.py not found. Attach Kaggle Dataset 'tetraft-code' with flat .py modules, "
        "or copy data.py into /kaggle/working."
    )

sys.path.insert(0, str(code_root))
print("Using code from:", code_root)
print("data.py:", code_root / "data.py")

In [ ]:
from data import build_fineweb_sample

OUT = Path("/kaggle/working/fineweb-50m")
OUT.mkdir(parents=True, exist_ok=True)

# 50M train + 0.5M val (PLAN default). Token counts are whitespace estimates.
meta = build_fineweb_sample(
    output_dir=OUT,
    max_train_tokens=50_000_000,
    max_val_tokens=500_000,
    seed=42,
    # Pin a HF revision string here once known, e.g. hf_revision="...",
)
print(meta)

In [ ]:
# Sanity check outputs
import json
from pathlib import Path

OUT = Path("/kaggle/working/fineweb-50m")
for name in ("train.jsonl", "val.jsonl", "sample_meta.json"):
    p = OUT / name
    assert p.is_file(), f"missing {p}"
    print(f"{name}: {p.stat().st_size / 1e6:.1f} MB")

with open(OUT / "sample_meta.json") as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))

# Peek first val line
with open(OUT / "val.jsonl") as f:
    row = json.loads(f.readline())
print("val sample keys:", row.keys())
print("val text preview:", row["text"][:200].replace("\n", " "))

## Save as Kaggle Dataset

1. **Save Version** (Save & Run All, or quick save if cells already ran).
2. Enable **output** in the save dialog.
3. Open the completed version → **New Dataset** from output (or Dataset → New → from notebook output).
4. Name: `tetraft-fineweb-edu-50m`.
5. Keep files under `fineweb-50m/`:
   - `train.jsonl`
   - `val.jsonl`
   - `sample_meta.json`

Next: attach `tetraft-code` + `tetraft-fineweb-edu-50m` and run `notebooks/run_smoke.ipynb`.